In [ ]:
import heapq
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import deque

# Helper Functions
def manhattan(cell1, cell2): return abs(cell1[0] - cell2[0]) + abs(cell1[1] - cell2[1])

def get_neighbors(grid, row, col):
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    neighbors = []
    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc
        if (0 <= new_row < len(grid) and 0 <= new_col < len(grid[0]) and grid[new_row][new_col] == 1):
            neighbors.append((new_row, new_col))
    return neighbors

def reconstruct_path(parent, start, goal):
    path = []
    current = goal
    while current != start:
        path.append(current)
        current = parent[current]
    path.append(start)
    path.reverse()
    return path

# Import/re-use grid generation logic here (omitted for brevity, assume simple grid for tests or re-use)
def create_basic_grid(rows=20, cols=24):
    grid = [[1 for _ in range(cols)] for _ in range(rows)]
    for r in range(rows):
        grid[r][0] = grid[r][cols-1] = 0
    for c in range(cols):
        grid[0][c] = grid[rows-1][c] = 0
    return grid

def astar(grid, start, goal):
    """
    A* Search — optimal + efficient using f(n) = g(n) + h(n).
    h(n) = Manhattan distance. Uses Min-Heap Priority Queue.
    Returns: (path, visited_order, nodes_explored)
    """
    h = lambda cell: manhattan(cell, goal)
    open_list = [(h(start), 0, start)]
    g_scores = {start: 0}
    parent = {start: None}
    visited_order = []
    closed_set = set()
    
    while open_list:
        f, g, cell = heapq.heappop(open_list)
        if cell in closed_set:
            continue
        closed_set.add(cell)
        visited_order.append(cell)
        r, c = cell
        hval = manhattan(cell, goal)
        print(f"A*: f(n) = g({g}) + h({hval}) = {f} for cell ({r}, {c})")
        
        if cell == goal:
            path = reconstruct_path(parent, start, goal)
            return path, visited_order, len(visited_order)
        
        for neighbor in get_neighbors(grid, r, c):
            new_g = g + 1
            if neighbor not in g_scores or new_g < g_scores[neighbor]:
                g_scores[neighbor] = new_g
                new_f = new_g + h(neighbor)
                parent[neighbor] = cell
                heapq.heappush(open_list, (new_f, new_g, neighbor))
    
    return [], visited_order, len(visited_order)


def hill_climbing(grid, start, goal, max_restarts=20):
    """
    Local Search — greedy move toward goal each step.
    Gets stuck at local maxima → uses random restarts to escape.
    Returns: (path, visited_order, nodes_explored, restarts)
    """
    current = start
    visited_order = []
    full_path = [current]
    restarts = 0
    open_cells = [(r, c) for r in range(len(grid)) 
                  for c in range(len(grid[0])) if grid[r][c] == 1]
    
    for _ in range(10000):
        if current == goal:
            return full_path, visited_order, len(visited_order), restarts
        
        visited_order.append(current)
        r, c = current
        neighbors = get_neighbors(grid, r, c)
        
        if not neighbors:
            break
            
        best = min(neighbors, key=lambda nb: manhattan(nb, goal))
        
        if manhattan(best, goal) < manhattan(current, goal):
            print(f"Hill Climbing: Moving to ({best[0]}, {best[1]}) — distance: {manhattan(best, goal)}")
            current = best
            full_path.append(current)
        else:
            if restarts >= max_restarts:
                break
            restart_cell = random.choice(open_cells)
            print(f"Hill Climbing: Stuck — Local Maximum! Restarting from ({restart_cell[0]}, {restart_cell[1]})")
            current = restart_cell
            restarts += 1
            full_path.append(current)
    
    return full_path, visited_order, len(visited_order), restarts

# Demo
grid = create_basic_grid()
s, g = (1, 1), (18, 22)
print("A* Execution:")
apath, avisited, anodes = astar(grid, s, g)
print("HC Execution:")
hpath, hvisited, hnodes, hrestart = hill_climbing(grid, s, g)